## Attacker

In [1]:
import numpy as np

class Attacker:
    """Base class for attackers."""
    def __init__(self, budget=0.3):
        self.budget = budget # Max feature manipulation budget

    def evade(self, classifier, X):
        raise NotImplementedError

class Level0Attacker(Attacker):
    """Level-0 Attacker (Random): Applies random feature manipulations."""
    def evade(self, classifier, X):
        X_evaded = X.copy()
        n_samples, n_features = X.shape
        max_flips = int(self.budget * n_features)

        for i in range(n_samples):
            num_flips = np.random.randint(1, max_flips + 1)
            flip_indices = np.random.choice(n_features, num_flips, replace=False)
            X_evaded[i, flip_indices] = 1 - X_evaded[i, flip_indices]

        return X_evaded

class Level1Attacker(Attacker):
    """Level-1 Attacker (Naive Best Response): Gradient-based evasion."""
    def evade(self, classifier, X):
        X_evaded = X.copy()
        # Assume linear classifier for simplicity (Logistic Regression)
        weights = getattr(classifier, 'coef_', None)
        if weights is None:
            return X_evaded
            
        weights = weights.flatten()
        n_samples, n_features = X.shape
        max_flips = int(self.budget * n_features)
        
        # Sort features by weight to find most negative impact (push towards ham)
        # For simplicity, we flip the top k most predictive spam features to 0
        target_features = np.argsort(weights)[-max_flips:]
        
        for i in range(n_samples):
            # Only flip features that are currently 1
            for idx in target_features:
                if X_evaded[i, idx] == 1:
                    X_evaded[i, idx] = 0
                    
        return X_evaded

class Level2Attacker(Attacker):
    """Level-2 Attacker (Anticipatory): Anticipates Level-1 defense."""
    def evade(self, classifier, X):
        # Placeholder for anticipatory logic
        # Typically involves training a shadow model and expecting defender adaptation
        return Level1Attacker(self.budget).evade(classifier, X)


## Defender

In [2]:
import numpy as np
from sklearn.linear_model import LogisticRegression

class Defender:
    def fit(self, X, y):
        raise NotImplementedError
    def predict(self, X):
        raise NotImplementedError

class StaticClassifier(Defender):
    def __init__(self):
        self.model = LogisticRegression(max_iter=1000)
        
    def fit(self, X, y):
        self.model.fit(X, y)
        
    def predict(self, X):
        return self.model.predict(X)

class BoundedRationalDefender(Defender):
    """
    Robust Level-k Defense algorithm (BRD).
    θ* = argmax_θ Σ_k p_k · U_D(θ, e_k(θ))
    """
    def __init__(self, beliefs={0: 0.1, 1: 0.6, 2: 0.3}, iterations=10, budget=0.3):
        self.beliefs = beliefs
        self.iterations = iterations
        self.budget = budget
        self.model = LogisticRegression(max_iter=1000)
        
    def fit(self, X, y):
        # Implement BRD Algorithm
        attackers = {
            0: Level0Attacker(self.budget),
            1: Level1Attacker(self.budget),
            2: Level2Attacker(self.budget)
        }
        
        # 1. Initialize θ_0 (standard training)
        self.model.fit(X, y)
        
        # Split spam and ham
        is_spam = (y == 1)
        X_spam, y_spam = X[is_spam], y[is_spam]
        X_ham, y_ham = X[~is_spam], y[~is_spam]
        
        # 2. Iterate T times
        for t in range(self.iterations):
            X_mix = []
            y_mix = []
            
            # Add ham (not manipulated)
            X_mix.append(X_ham)
            y_mix.append(y_ham)
            
            # a. For each level k
            for k, prob in self.beliefs.items():
                if prob == 0: continue
                # i / ii. Simulate attacker and generate adversarial examples
                X_k_evaded = attackers[k].evade(self.model, X_spam.copy())
                
                # We conceptually weigh datasets by repeating or using sample_weight
                # For simplicity, we just aggregate and use sample weights dynamically later
                X_mix.append(X_k_evaded)
                y_mix.append(y_spam)
                
            X_train = np.vstack(X_mix)
            y_train = np.hstack(y_mix)
            
            # Construct sample weights based on beliefs
            weights = np.ones(len(y_train))
            idx = len(y_ham)
            for k, prob in self.beliefs.items():
                if prob == 0: continue
                chunk_size = len(X_spam)
                weights[idx:idx+chunk_size] = prob
                idx += chunk_size
                
            # c. Train classifier on mixed dataset
            self.model.fit(X_train, y_train, sample_weight=weights)
            
    def predict(self, X):
        return self.model.predict(X)


## Classifier


In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression

class ContentClassifier:
    """Wrapper for base classifiers."""
    def __init__(self, model_type='logistic'):
        self.model_type = model_type
        if self.model_type == 'logistic':
            self.model = LogisticRegression(max_iter=1000)
        else:
            raise ValueError(f"Model type {model_type} not supported yet.")
            
    def fit(self, X, y):
        self.model.fit(X, y)
        
    def predict(self, X):
        return self.model.predict(X)
        
    def predict_proba(self, X):
        return self.model.predict_proba(X)


## Data Preprocessing

In [4]:
import os
import urllib.request
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def download_and_extract(data_dir="data/enron_spam"):
    """
    Downloads a sample Enron / Spam dataset CSV if it doesn't already exist.
    """
    os.makedirs(data_dir, exist_ok=True)
    csv_path = os.path.join(data_dir, "spam_data.csv")
    
    if os.path.exists(csv_path):
        return csv_path
        
    print(f"Downloading spam dataset to {csv_path}...")
    try:
        # We download a well-known preprocessed SMS / Email CSV to prototype on.
        url = "https://raw.githubusercontent.com/amankharwal/Website-data/master/spam.csv"
        urllib.request.urlretrieve(url, csv_path)
        print("Download complete.")
        return csv_path
    except Exception as e:
        print(f"Failed to download: {e}")
        return None

def extract_features(csv_path, max_features=80):
    """
    Extracts features from the text data using TF-IDF.
    Matches the 80-dimensional feature space described in the paper.
    """
    try:
        df = pd.read_csv(csv_path, encoding='latin-1')
        
        # Determine correct columns if they differ from the Kaggle default 'v1', 'v2'
        if 'v1' in df.columns:
            df = df[['v1', 'v2']]
            df.columns = ['label', 'text']
        elif 'label' in df.columns and 'text' in df.columns:
            df = df[['label', 'text']]
            
        # Convert labels: 'spam' -> 1, 'ham' -> 0
        y = np.where(df['label'] == 'spam', 1, 0)
        
        # TF-IDF Vectorization
        vectorizer = TfidfVectorizer(max_features=max_features, stop_words='english')
        X = vectorizer.fit_transform(df['text']).toarray()
        feature_names = vectorizer.get_feature_names_out()
        
        return X, y, feature_names
    except Exception as e:
        print(f"Feature extraction failed: {e}")
        return None, None, None

def load_real_data(max_features=80):
    csv_path = download_and_extract()
    if csv_path:
        X, y, feature_names = extract_features(csv_path, max_features=max_features)
        return X, y, feature_names
    return None, None, None

if __name__ == "__main__":
    X, y, features = load_real_data()
    if X is not None:
        print(f"Preprocessing successful: Formed array of shape {X.shape}.")
        print(f"First 5 features: {features[:5]}")

Preprocessing successful: Formed array of shape (5572, 80).
First 5 features: ['amp' 'ask' 'babe' 'cash' 'claim']


## Evaluation

In [5]:
def evaluate_predictions(y_true, y_pred, C_fp=10, C_fn=1):
    tp = sum((y_true == 1) & (y_pred == 1))
    fp = sum((y_true == 0) & (y_pred == 1))
    tn = sum((y_true == 0) & (y_pred == 0))
    fn = sum((y_true == 1) & (y_pred == 0))
    
    detection_rate = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1 = 2 * (precision * detection_rate) / (precision + detection_rate) if (precision + detection_rate) > 0 else 0
    
    # Cost-weighted calculation
    total_cost = (fp * C_fp) + (fn * C_fn)
    
    return {
        'Detection Rate': detection_rate,
        'False Positive Rate': fpr,
        'F1 Score': f1,
        'Total Cost': total_cost
    }


## Experiments


In [ ]:
import os
import sys
import yaml
import numpy as np

def load_config(config_path):
    with open(config_path, 'r') as f:
        return yaml.safe_load(f)

def generate_synthetic_data(n_samples=1000, n_features=80):
    # Dummy synthetic data generator mimicking typical spam/ham distributions
    X = np.random.binomial(1, 0.2, (n_samples, n_features))
    # Let's say first 10 features are strong spam predictors
    y = np.zeros(n_samples)
    spam_idx = np.random.choice(n_samples, n_samples // 2, replace=False)
    y[spam_idx] = 1
    X[spam_idx, :10] = np.random.binomial(1, 0.8, (len(spam_idx), 10))
    return X, y

def run_experiment():
    print("Running BRD evaluation experiment...")
    config_path = os.path.join('config.yaml')
    config = load_config(config_path)
    
    # 1. Load Data
    print("Loading real TF-IDF text features...")
    X, y, feature_names = load_real_data(max_features=80)
    
    # Randomly shuffle data to ensure mixed train/test
    indices = np.random.permutation(len(X))
    X, y = X[indices], y[indices]
    
    n_train = int(len(X) * config['data']['split']['train'])
    X_train, y_train = X[:n_train], y[:n_train]
    X_test, y_test = X[n_train:], y[n_train:]
    
    # 2. Train Static Classifier
    print("Training Static Classifier...")
    static = StaticClassifier()
    static.fit(X_train, y_train)
    static_acc = np.mean(static.predict(X_test) == y_test)
    print(f"Static Classifier Init Acc: {static_acc:.4f}")
    
    # 3. Train BRD Classifier
    print("Training Bounded Rational Defender (BRD)...")
    beliefs = {
        0: config['brd_params']['belief_distribution']['k0'],
        1: config['brd_params']['belief_distribution']['k1'],
        2: config['brd_params']['belief_distribution']['k2']
    }
    brd = BoundedRationalDefender(beliefs=beliefs, iterations=config['brd_params']['iterations'], budget=config['data']['budget'])
    brd.fit(X_train, y_train)
    brd_acc = np.mean(brd.predict(X_test) == y_test)
    print(f"BRD Init Acc: {brd_acc:.4f}")
    
    # Simulate Attack Evaluation
    print("Evaluating against Level-1 Attack on Test Set...")
    attacker = Level1Attacker(budget=config['data']['budget'])
    is_spam = (y_test == 1)
    X_test_evaded = X_test.copy()
    X_test_evaded[is_spam] = attacker.evade(static.model, X_test[is_spam])
    
    static_evaded_acc = np.mean(static.predict(X_test_evaded) == y_test)
    print(f"Static Classifier Evaded Acc: {static_evaded_acc:.4f}")
    
    X_test_brd_evaded = X_test.copy()
    X_test_brd_evaded[is_spam] = attacker.evade(brd.model, X_test[is_spam])
    brd_evaded_acc = np.mean(brd.predict(X_test_brd_evaded) == y_test)
    print(f"BRD Evaded Acc: {brd_evaded_acc:.4f}")
    
    print("-" * 40)
    print(f"Improvement under attack: {(brd_evaded_acc - static_evaded_acc) * 100:.2f}%")
    
    print("\nDetailed Evaluation Metrics (Level-1 Attack):")
    static_eval = evaluate_predictions(y_test, static.predict(X_test_evaded))
    brd_eval = evaluate_predictions(y_test, brd.predict(X_test_brd_evaded))
    
    print(f"Static Classifier -> F1: {static_eval['F1 Score']:.4f}, FPR: {static_eval['False Positive Rate']:.4f}, Cost: {static_eval['Total Cost']}")
    print(f"BRD Classifier    -> F1: {brd_eval['F1 Score']:.4f}, FPR: {brd_eval['False Positive Rate']:.4f}, Cost: {brd_eval['Total Cost']}")
    
if __name__ == "__main__":
    run_experiment()


Running BRD evaluation experiment...
Loading real TF-IDF text features...
Training Static Classifier...
Static Classifier Init Acc: 0.9507
Training Bounded Rational Defender (BRD)...
BRD Init Acc: 0.9507
Evaluating against Level-1 Attack on Test Set...
Static Classifier Evaded Acc: 0.9426
BRD Evaded Acc: 0.9448
----------------------------------------
Improvement under attack: 0.22%

Detailed Evaluation Metrics (Level-1 Attack):
Static Classifier -> F1: 0.7470, FPR: 0.0083, Cost: 272
BRD Classifier    -> F1: 0.7535, FPR: 0.0052, Cost: 213
